# Acquirium client basics

This is a guide to show the basics of acquirium client.

To get support reach out to [Mete](mailto:saka@mines.edu)!

## Initial setup:
1. create and activate a fresh virtual environment: `python -m venv .venv && source .venv/bin/activate` (Linux/macOS) or `.venv\Scripts\activate` (Windows). **Requires Python 3.12**+.
    - Alternatively you can use [uv package manager] with `uv init --python 3.12`
2. Install Acquirium from PyPI: `pip install acquirium[watertap]`.
    - Alternatively: `uv add acquirium[watertap]`
3. Start the server (plus any drivers listed in the config): 
    - `acquirium server --config deployments/WATERTAP/acquirium.toml`.
    - Alternatively, `uv run acquirium server --config deployments/WATERTAP/acquirium.toml`.
4. Verify it's up by opening [`http://localhost:8000/docs`](http://localhost:8000/docs) (or whichever host/port your config sets) in a browser.
    - Alternatively: `curl localhost:8000/health` from another terminal
    - Or using Python session or notebook, run:
    ```
    from acquirium import Acquirium 
    acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)
    ```

## Loading required libraries and acquirium client

Acquirium is shipped with it's client to connect the server and help users to use the server with a python interface.

Acquirium client can be loaded and initiated with:

In [42]:
from datetime import datetime, timedelta, timezone
from acquirium import Acquirium

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

## Find entities by class
`_class` accepts a URI or a natural-language string. `alias` names the node for later reference.

In [43]:
q = acq.find_entity(_class="Pump", alias="pump")
_ = q.metadata_head()

Metadata First
   10 Rows    
┏━━━━━━━━━━━━┓
┃ pump       ┃
┡━━━━━━━━━━━━┩
│ wbs:P1     │
│ wbs:P2     │
│ wbs:intake │
└────────────┘

## Follow relationships
`find_related` adds a neighbour reachable within `hops`. Use `direction="upstream"/"downstream"` to walk S223 connections.

In [44]:
q = (
    acq.find_entity(_class="Pump", alias="pump")
       .find_related(_class="Tank", alias="tank", _from="pump", hops=1)
)
q.show_query_graph()
_ = q.metadata_head()

QUERY GRAPH

Nodes:
  0 [pump]  class=http://data.ashrae.org/standard223#Pump
  2 [tank]  class=urn:nawi-water-ontology#Tank

Edges:
  pump --(*, hops=1)--> tank

Data nodes: (none)

Current pointer: tank



           Metadata First 10 Rows            
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ tank                         ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:intake │ wbs:ferric-chloride-addition │
└────────────┴──────────────────────────────┘

## Attach data nodes
`find_data` adds the observable/actuatable properties of the current node. `find_all_data` does it for every entity in the graph.


In [45]:
q = acq.find_entity(_class="Pump", alias="pump").find_data()
_ = q.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ pump_data                       ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:P1     │ wbs:P1-efficiency               │
│ wbs:P1     │ wbs:P1-efficiency               │
└────────────┴─────────────────────────────────┘

In [46]:
q_all = acq.find_all_data()
_ =q_all.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:storage-tank-3-out-flow-rate             │
│ wbs:RO-out-flow-mass-water                   │
│ wbs:P1-out-pressure                          │
│ wbs:conn-cartridge-filtration-to-S1-pressure │
│ wbs:intake-in-tds-concentration              │
│ wbs:PXR-brine-out-flow-mass-tds              │
│ wbs:P1-mechanical-power                      │
│ wbs:intake-in-tds-concentration              │
│ wbs:storage-tank-3-out-flow-rate             │
│ wbs:RO-out-retentate-flow-mass-tds           │
└──────────────────────────────────────────────┘

## Filter data nodes
Filters apply to the bound data nodes. Strings are resolved via the text matcher.

In [47]:
q = (
    acq.find_all_data()
       .filter_by_quantity_kind("Pressure")
)
_ =q.metadata_head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:conn-cartridge-filtration-to-S1-pressure │
│ wbs:conn-cartridge-filtration-to-S1-pressure │
│ wbs:PXR-brine-out-pressure                   │
│ wbs:PXR-brine-out-pressure                   │
│ wbs:RO-in-pressure                           │
│ wbs:RO-in-pressure                           │
│ wbs:RO-out-retentate-pressure                │
│ wbs:RO-out-retentate-pressure                │
│ wbs:RO-out-pressure                          │
│ wbs:RO-out-pressure                          │
└──────────────────────────────────────────────┘

In [48]:
q = (
    acq.find_all_data()
       .filter_by_unit("KG/s")
)
_ =q.metadata_head()

                Metadata First 10 Rows                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:PXR-brine-out-flow-mass-water                   │
│ wbs:PXR-brine-out-flow-mass-water                   │
│ wbs:RO-in-flow-mass-water                           │
│ wbs:RO-in-flow-mass-water                           │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-water │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-water │
│ wbs:RO-out-flow-mass-tds                            │
│ wbs:RO-out-flow-mass-tds                            │
│ wbs:RO-out-retentate-flow-mass-tds                  │
│ wbs:RO-out-retentate-flow-mass-tds                  │
└─────────────────────────────────────────────────────┘

In [49]:
q = (
    acq.find_all_data()
        .filter_by_substance("constituent Salt")
        .filter_by_unit("KG/s")
)
_ =q.metadata_head()

               Metadata First 10 Rows                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ 0                                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:RO-out-flow-mass-tds                          │
│ wbs:RO-out-flow-mass-tds                          │
│ wbs:RO-out-retentate-flow-mass-tds                │
│ wbs:RO-out-retentate-flow-mass-tds                │
│ wbs:RO-in-flow-mass-tds                           │
│ wbs:RO-in-flow-mass-tds                           │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds │
│ wbs:conn-cartridge-filtration-to-S1-flow-mass-tds │
│ wbs:PXR-brine-out-flow-mass-tds                   │
│ wbs:PXR-brine-out-flow-mass-tds                   │
└───────────────────────────────────────────────────┘

## Inspect the query
`show_query_graph` prints the node/edge structure; `to_sparql` returns the compiled SPARQL; `metadata` returns the full result as a polars DataFrame.

In [50]:
q.show_query_graph()

QUERY GRAPH

Nodes:
  0 [0] [DATA]  class=*

Edges:

Data nodes:
  0 [0]  filters={http://data.ashrae.org/standard223#ofSubstance=['urn:nawi-water-ontology#Constituent-Salt'], http://qudt.org/schema/qudt/hasUnit=['http://qudt.org/vocab/unit/KiloGM-PER-SEC']}}

Current pointer: 0



In [51]:
print(q.to_sparql())

SELECT DISTINCT ?v0 ?ext0 ?unit0 ?extunit0
WHERE {
  ?v0 <https://brickschema.org/schema/Brick/ref#hasExternalReference> ?ext0 .
  OPTIONAL { ?v0 <http://qudt.org/schema/qudt/hasUnit> ?unit0 . }
  OPTIONAL { ?ext0 <http://qudt.org/schema/qudt/hasUnit> ?extunit0 . }
  { { ?v0 <http://data.ashrae.org/standard223#ofSubstance> <urn:nawi-water-ontology#Constituent-Salt> . } }
  { { ?v0 <http://qudt.org/schema/qudt/hasUnit> <http://qudt.org/vocab/unit/KiloGM-PER-SEC> . } }
}


In [52]:
df_meta = q.metadata()
df_meta

0
str
"""wbs:RO-out-flow-mass-tds"""
"""wbs:RO-out-flow-mass-tds"""
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO-in-flow-mass-tds"""
"""wbs:RO-in-flow-mass-tds"""
"""wbs:conn-cartridge-filtration-…"
"""wbs:conn-cartridge-filtration-…"
"""wbs:PXR-brine-out-flow-mass-td…"


## Pull timeseries
`dataframe` returns a polars frame. `shape="wide"` puts each data node in its own column, `"narrow"` is long-form. `latest_data` is a shortcut for the most recent point.

In [63]:
end = datetime.now(tz=timezone.utc)
start = end - timedelta(minutes=10)

df = q.dataframe(start=start, end=end, shape="wide", cast_value="float")
df.head()

time,wbs:RO-in-flow-mass-tds,wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,wbs:RO-out-flow-mass-tds,wbs:PXR-brine-out-flow-mass-tds,wbs:RO-out-retentate-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-07-01 00:13:18.140063 UTC,11.502723,11.502723,0.028782,11.473941,11.473941
2026-07-01 00:13:27.210202 UTC,11.502723,11.502723,0.028455,11.474268,11.474268
2026-07-01 00:13:33.844473 UTC,11.502723,11.502723,0.028293,11.47443,11.47443
2026-07-01 00:15:20.114227 UTC,11.847797,11.847797,0.028091,11.819706,11.819706
2026-07-01 00:16:15.624634 UTC,11.962813,11.962813,0.028025,11.934788,11.934788


In [71]:
q.latest_data(limit=2)

time,wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,wbs:RO-in-flow-mass-tds,wbs:RO-out-flow-mass-tds,wbs:RO-out-retentate-flow-mass-tds,wbs:PXR-brine-out-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-07-01 00:16:22.710913 UTC,12.307862,12.307862,0.027831,12.280031,12.280031
2026-07-01 00:16:36.510997 UTC,12.422878,12.422878,0.027768,12.39511,12.39511


## Structured access via DataObject
`data()` returns an object keyed by alias for quick lookups.

In [72]:
import polars as pl


data = q.data(start=start, end=end, cast_value="float")
data.dataframe().cast(pl.Float64)

time,0__wbs:RO-out-flow-mass-tds,0__wbs:RO-in-flow-mass-tds,0__wbs:RO-out-retentate-flow-mass-tds,0__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,0__wbs:PXR-brine-out-flow-mass-tds
f64,f64,f64,f64,f64,f64
1.7829e15,0.028782,11.502723,11.473941,11.502723,11.473941
1.7829e15,0.028455,11.502723,11.474268,11.502723,11.474268
1.7829e15,0.028293,11.502723,11.47443,11.502723,11.47443
1.7829e15,0.028091,11.847797,11.819706,11.847797,11.819706
1.7829e15,0.028025,11.962813,11.934788,11.962813,11.934788


## Inspect units
`units()` returns the effective QUDT unit URI per data alias.

In [73]:
data.units()

{'0': 'http://qudt.org/vocab/unit/KiloGM-PER-SEC'}

## Convert units
`convert_to(target)` accepts any QUDT-recognized identifier (URI, label, symbol, UCUM code). The returned DataObject has values converted and `units()` updated.

In [74]:
data = data.convert_to("kg/min")
data.dataframe().head()

time,0__wbs:RO-out-retentate-flow-mass-tds,0__wbs:PXR-brine-out-flow-mass-tds,0__wbs:RO-in-flow-mass-tds,0__wbs:conn-cartridge-filtration-to-S1-flow-mass-tds,0__wbs:RO-out-flow-mass-tds
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-07-01 00:13:18.140063 UTC,688.436457,688.436457,690.163382,690.163382,1.726925
2026-07-01 00:13:27.210202 UTC,688.456101,688.456101,690.163382,690.163382,1.707281
2026-07-01 00:13:33.844473 UTC,688.465773,688.465773,690.163382,690.163382,1.697609
2026-07-01 00:15:20.114227 UTC,709.182342,709.182342,710.867803,710.867803,1.685461
2026-07-01 00:16:15.624634 UTC,716.087279,716.087279,717.768778,717.768778,1.681499


### Systems

Systems are logical groupings of equipment and junctions (and other systems) in S223 ontology (parent ontology of WaTr)

The systems in the model are:

In [75]:
def list_systems():
    q = acq.find_entity(_class = "System", alias = "Systems")
    return q.metadata()
list_systems()

Systems
str
"""wbs:pretreatment-system"""
"""wbs:desalination-system"""
"""wbs:posttreatment-system"""
"""wbs:seawater-ro-plant"""


The systems are hierarchically organized as:

In [76]:
def list_systems_hier():
    q = acq.find_entity(_class = "System", alias = "Systems")
    q = q.find_related(_class = "System", predicates = ['hasMember'], alias = "Subsystem")
    q.metadata_head()
list_systems_hier()

               Metadata First 10 Rows               
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Systems               ┃ Subsystem                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:seawater-ro-plant │ wbs:pretreatment-system  │
│ wbs:seawater-ro-plant │ wbs:desalination-system  │
│ wbs:seawater-ro-plant │ wbs:posttreatment-system │
└───────────────────────┴──────────────────────────┘

We can see how many equipment we have in each system:

In [77]:
def list_equipment_by_system(hops = 1):
        q = acq.find_entity(_class = "System", alias = "Systems")
        q = q.find_related(_class = "Equipment", predicates = ['hasMember'], alias = "Equipment", hops = hops, multi_hop_predicates = True)
        q_df = q.metadata()
        return q_df.group_by("Systems").agg(pl.col("Equipment").count().alias("equipment_count")).sort("equipment_count", descending=True)

list_equipment_by_system()

Systems,equipment_count
str,u32
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


These are the number of equipment directly a member of these systems.

If we increase the hops, you'll see total number equipments in each system

In [78]:
list_equipment_by_system(3)

Systems,equipment_count
str,u32
"""wbs:seawater-ro-plant""",18
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


Let's find the pumps in a specific system:


In [79]:
def list_equipment_in_system(system, equipment):
    q = acq.find_entity(uri=system, alias = "system").find_related(_class=equipment, alias = "equipment", hops=1)
    q.metadata_head()

list_equipment_in_system('wbs:pretreatment-system', 'pump')

         Metadata First 10 Rows         
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ system                  ┃ equipment  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ wbs:pretreatment-system │ wbs:intake │
└─────────────────────────┴────────────┘

Let's find all the pumps and their data

In [80]:
def all_pumps_and_their_data():
    q = acq.find_entity(_class="pump", alias="pump").find_all_data()
    q.metadata_head()
    return q.data(limit=10).dataframe()

all_pumps_and_their_data().head()

             Metadata First 10 Rows             
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ pump       ┃ pump_data                       ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:P1     │ wbs:P1-out-pressure             │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-tss-concentration │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-flow-rate         │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:intake │ wbs:intake-in-tds-concentration │
│ wbs:P1     │ wbs:P1-efficiency               │
│ wbs:P1     │ wbs:P1-efficiency               │
└────────────┴─────────────────────────────────┘

time,pump_data__wbs:P1-efficiency,pump_data__wbs:P2-mechanical-power,pump_data__wbs:P2-efficiency,pump_data__wbs:P1-mechanical-power,pump_data__wbs:intake-in-tds-concentration,pump_data__wbs:intake-in-flow-rate,pump_data__wbs:P1-out-pressure,pump_data__wbs:intake-in-tss-concentration
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64
2026-07-01 00:13:18.140063 UTC,0.8,149579.616304,0.8,1.2561e6,33.699458,0.341333,7e6,0.038317
2026-07-01 00:13:27.210202 UTC,0.8,147512.002088,0.8,1.2406e6,33.699458,0.341333,7e6,0.038317
2026-07-01 00:13:33.844473 UTC,0.8,146752.494201,0.8,1.2332e6,33.699458,0.341333,7e6,0.038317
2026-07-01 00:15:20.114227 UTC,0.8,156542.546383,0.8,1.2468e6,33.699458,0.351572,7e6,0.038317
2026-07-01 00:16:15.624634 UTC,0.8,159902.69015,0.8,1.2512e6,33.699458,0.354985,7e6,0.038317


Let's find all the data generating entites within a system:

In [81]:
def find_all_sensors(system):
    q = (acq.find_entity(uri=system, alias="backwash")
         .find_related(_class="equipment", alias="equipment",predicates=['hasMember'], hops=1)
         .find_data(alias = "sensors"))
    q_df = q.metadata(include_internals=True)
    q_df = q_df.drop([pl.col('backwash'),pl.col('sensors_ref'),pl.col('extunit4')])
    return q_df

system = 'wbs:pretreatment-system'
find_all_sensors(system)

equipment,sensors,unit4
str,str,str
"""wbs:intake""","""wbs:intake-in-tss-concentratio…","""unit:MilliGM-PER-L"""
"""wbs:intake""","""wbs:intake-in-tss-concentratio…","""unit:MilliGM-PER-L"""
"""wbs:intake""","""wbs:intake-in-flow-rate""","""None"""
"""wbs:intake""","""wbs:intake-in-flow-rate""","""None"""
"""wbs:intake""","""wbs:intake-in-tds-concentratio…","""unit:KiloGM-PER-M3"""
"""wbs:intake""","""wbs:intake-in-tds-concentratio…","""unit:KiloGM-PER-M3"""
